# 02_import_rubq_mkqa_ru

Первый ingestion-ноутбук для внешних источников русскоязычного бенчмарка multi-hop / compositional QA.

## Что делает этот ноутбук
1. Загружает **RuBQ 2.0** и **MKQA**.
2. Оставляет только русский слой и приводит оба источника к **единой схеме**.
3. Разделяет примеры по ролям:
   - **direct_ru_candidate** — хорошие кандидаты на прямое включение или близкий перенос;
   - **reconstruction_candidate** — кандидаты на пересборку под твой core truth layer;
   - **wording_donor** — доноры русских формулировок и answer normalization.
4. Экспортирует результаты в JSONL / CSV.
5. Готовит сводку и удобные выборки для следующих ноутбуков.

## Важное упрощение этой версии
В этой версии ноутбук **не делает**:
- dedup,
- sanitization / quality filtering,
- сравнение против твоего уже собранного core-layer.

То есть это чистый **import + normalization + export**.


## Единая схема записи

Каждая нормализованная запись экспортируется примерно в таком виде:

```json
{
  "benchmark_id": "ext_rubq2_dev_000004",
  "source_dataset": "RuBQ_2.0",
  "source_split": "dev",
  "source_example_id": "4",
  "import_mode": "direct_ru",
  "use_role": "direct_ru_candidate",
  "question_ru": "...",
  "question_en_original": "...",
  "domain": null,
  "reasoning_type": ["multi_hop", "ranking"],
  "hop_count": 2,
  "constraints_struct": {...},
  "formal_query": "SELECT ...",
  "program": null,
  "gold_answers": [...],
  "gold_qids": ["Q298"],
  "supporting_facts": [],
  "supporting_entities": ["Q14452", "P17"],
  "evidence_sources": [...],
  "is_answerable": true,
  "expected_cardinality": 1,
  "paraphrase_cluster_id": null,
  "difficulty": "unknown",
  "manual_audit_status": "pending"
}
```

> Важно: этот ноутбук **не пытается окончательно решить**, что войдёт в финальный бенч.  
> Его цель — сделать чистый **первый внешний ingestion layer** без dedup и санитичека, чтобы дальше ты мог сам отдельно провести аудит, фильтрацию и merge.

In [1]:

# Если запускаешь в чистом окружении, раскомментируй:
%pip install -q datasets pandas pyarrow requests matplotlib

import csv
import gzip
import hashlib
import json
import math
import os
import re
import textwrap
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import pandas as pd

try:
    from datasets import load_dataset
except Exception:
    load_dataset = None


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# =========================
# CONFIG
# =========================

PROJECT_ROOT = Path.cwd()
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts_external_stage1"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

EXPORT_JSONL_DIR = ARTIFACTS_DIR / "jsonl"
EXPORT_JSONL_DIR.mkdir(parents=True, exist_ok=True)

EXPORT_CSV_DIR = ARTIFACTS_DIR / "csv"
EXPORT_CSV_DIR.mkdir(parents=True, exist_ok=True)

REPORTS_DIR = ARTIFACTS_DIR / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)


RUBQ_HF_NAME = "d0rj/RuBQ_2.0"
MKQA_HF_NAME = "apple/mkqa"

RUBQ_RAW_URLS = {
    "dev": "https://raw.githubusercontent.com/vladislavneon/RuBQ/master/RuBQ_2.0/RuBQ_2.0_dev.json",
    "test": "https://raw.githubusercontent.com/vladislavneon/RuBQ/master/RuBQ_2.0/RuBQ_2.0_test.json",
}
MKQA_RAW_URL = "https://github.com/apple/ml-mkqa/raw/main/dataset/mkqa.jsonl.gz"

# Политика отбора
KEEP_RUBQ_COMPLEX_ONLY_FOR_DIRECT = True
MIN_MKQA_COMPLEXITY_SCORE_FOR_RECONSTRUCTION = 2

# Для отчётов
MAX_AUDIT_ROWS_PER_GROUP = 200

SEED_NOTE = 42  # здесь только для протокола воспроизводимости
print(f"[config] artifacts dir: {ARTIFACTS_DIR.resolve()}")

[config] artifacts dir: /Users/matvey/Desktop/artifacts_external_stage1


In [4]:

# =========================
# БАЗОВЫЕ УТИЛИТЫ
# =========================

def normalize_spaces(text: Optional[str]) -> str:
    if text is None:
        return ""
    text = str(text).replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def normalize_for_dedup(text: Optional[str]) -> str:
    text = normalize_spaces(text).lower().replace("ё", "е")
    text = re.sub(r"[«»\"'`]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def stable_hash(text: str, n: int = 16) -> str:
    return hashlib.md5(text.encode("utf-8")).hexdigest()[:n]

def extract_qid_from_uri(value: Optional[str]) -> Optional[str]:
    if not value:
        return None
    m = re.search(r"/(Q\d+)$", str(value))
    return m.group(1) if m else None

def extract_pid_from_prop(value: Optional[str]) -> Optional[str]:
    if not value:
        return None
    m = re.search(r"(P\d+)$", str(value))
    return m.group(1) if m else None

def write_jsonl(records: List[Dict[str, Any]], path: Path) -> None:
    with path.open("w", encoding="utf-8") as f:
        for row in records:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def save_csv(records: List[Dict[str, Any]], path: Path) -> None:
    if not records:
        pd.DataFrame().to_csv(path, index=False)
        return
    pd.DataFrame(records).to_csv(path, index=False)

def to_list(x: Any) -> List[Any]:
    if x is None:
        return []
    if isinstance(x, list):
        return x
    return [x]

def unique_preserve_order(items: Iterable[Any]) -> List[Any]:
    seen = set()
    out = []
    for item in items:
        key = json.dumps(item, ensure_ascii=False, sort_keys=True) if isinstance(item, (dict, list)) else item
        if key not in seen:
            seen.add(key)
            out.append(item)
    return out

def flatten_str_list(items: Iterable[Any]) -> List[str]:
    out = []
    for x in items:
        if x is None:
            continue
        if isinstance(x, list):
            out.extend(flatten_str_list(x))
        else:
            val = normalize_spaces(str(x))
            if val:
                out.append(val)
    return unique_preserve_order(out)

In [5]:

# =========================
# REASONING TAXONOMY
# =========================

RUBQ_TAG_TO_REASONING = {
    "1-hop": "single_hop",
    "0-hop": "single_hop",
    "multi-hop": "multi_hop",
    "multi-constraint": "multi_constraint",
    "qualifier-answer": "qualifier_answer",
    "qualifier-constraint": "qualifier_constraint",
    "reverse": "reverse_relation",
    "count": "count",
    "ranking": "ranking",
    "exclusion": "negation_or_exclusion",
    "duration": "temporal_or_duration",
    "no_answer": "no_answer",
}

MKQA_ANSWER_TYPE_NAMES = [
    "entity",
    "long_answer",
    "unanswerable",
    "date",
    "number",
    "number_with_unit",
    "short_phrase",
    "binary",
]

def rubq_reasoning_types(tags: List[str]) -> List[str]:
    types = []
    for tag in tags or []:
        if tag in RUBQ_TAG_TO_REASONING:
            types.append(RUBQ_TAG_TO_REASONING[tag])
    return unique_preserve_order(types)

def rubq_hop_count(tags: List[str]) -> Optional[int]:
    tags = set(tags or [])
    if "multi-hop" in tags:
        return 2
    if "1-hop" in tags:
        return 1
    if "0-hop" in tags:
        return 0
    return None

COMPLEX_RUBQ_TAGS = {
    "multi-hop",
    "multi-constraint",
    "qualifier-answer",
    "qualifier-constraint",
    "reverse",
    "count",
    "ranking",
    "exclusion",
    "duration",
    "no_answer",
}

def has_rubq_complexity(tags: List[str]) -> bool:
    return any(tag in COMPLEX_RUBQ_TAGS for tag in (tags or []))

RUSSIAN_COMPLEXITY_PATTERNS = {
    "negation_or_exclusion": [
        r"\bкроме\b",
        r"\bне\b",
        r"\bисключая\b",
        r"\bбез\b",
    ],
    "comparison_or_superlative": [
        r"\bсам(ый|ая|ое|ые)\b",
        r"\bбольше\b",
        r"\bменьше\b",
        r"\bнаибол",
        r"\bнаименьш",
        r"\bперв(ый|ая|ое|ые)\b",
        r"\bпоследн(ий|яя|ее|ие)\b",
    ],
    "temporal_or_duration": [
        r"\bдо\b",
        r"\bпосле\b",
        r"\bмежду\b",
        r"\bв период\b",
        r"\bгод",
        r"\bвеке\b",
        r"\bвек\b",
        r"\bдата\b",
        r"\bкогда\b",
    ],
    "set_intersection_or_multi_constraint": [
        r"\bи\b",
        r"\bа также\b",
        r"\bодновременно\b",
        r"\bиз .* и .*\b",
        r"\bкотор(ый|ая|ое|ые).* и .*\b",
    ],
    "count": [
        r"\bсколько\b",
        r"\bчисло\b",
        r"\bколичество\b",
    ],
}

def detect_mkqa_reasoning_types(question_ru: str) -> List[str]:
    q = " " + normalize_for_dedup(question_ru) + " "
    found = []
    for reasoning_type, patterns in RUSSIAN_COMPLEXITY_PATTERNS.items():
        if any(re.search(p, q) for p in patterns):
            found.append(reasoning_type)
    # Базовая эвристика: если нет ничего сложного, считаем это simple factual wording donor.
    if not found:
        found.append("single_hop_or_simple_open_qa")
    return unique_preserve_order(found)

def mkqa_complexity_score(question_ru: str, answers_ru: List[Dict[str, Any]]) -> int:
    score = 0
    reasoning_types = detect_mkqa_reasoning_types(question_ru)
    if "single_hop_or_simple_open_qa" not in reasoning_types:
        score += len(reasoning_types)
    if len(answers_ru) > 1:
        score += 1
    # поощряем списковые / многозначные ответы
    entity_answer_count = sum(1 for a in answers_ru if a.get("answer_type") == "entity")
    if entity_answer_count > 1:
        score += 1
    return score

In [6]:

# =========================
# ЗАГРУЗКА RuBQ 2.0
# =========================

def _load_rubq_from_hf() -> Dict[str, List[Dict[str, Any]]]:
    if load_dataset is None:
        raise RuntimeError("datasets library is not available")

    print(f"[rubq] trying Hugging Face dataset: {RUBQ_HF_NAME}")
    ds = load_dataset(RUBQ_HF_NAME)
    out = {
        "dev": [dict(x) for x in ds["dev"]],
        "test": [dict(x) for x in ds["test"]],
    }
    print(f"[rubq] loaded from HF -> dev={len(out['dev'])}, test={len(out['test'])}")
    return out

def _load_rubq_from_raw() -> Dict[str, List[Dict[str, Any]]]:
    import requests

    print("[rubq] HF loading failed, fallback to raw GitHub URLs")
    out = {}
    for split, url in RUBQ_RAW_URLS.items():
        print(f"[rubq] downloading {split} from {url}")
        response = requests.get(url, timeout=120)
        response.raise_for_status()
        out[split] = response.json()
        print(f"[rubq] loaded {split}: {len(out[split])}")
    return out

def load_rubq() -> Dict[str, List[Dict[str, Any]]]:
    try:
        return _load_rubq_from_hf()
    except Exception as e:
        print(f"[rubq] HF load failed: {type(e).__name__}: {e}")
        return _load_rubq_from_raw()

rubq_raw = load_rubq()

[rubq] trying Hugging Face dataset: d0rj/RuBQ_2.0


Generating dev split: 100%|██████████| 580/580 [00:00<00:00, 84389.51 examples/s]


[rubq] loaded from HF -> dev=580, test=2330


In [7]:

# =========================
# ЗАГРУЗКА MKQA
# =========================

def _load_mkqa_from_hf() -> List[Dict[str, Any]]:
    if load_dataset is None:
        raise RuntimeError("datasets library is not available")

    print(f"[mkqa] trying Hugging Face dataset: {MKQA_HF_NAME}")
    ds = load_dataset(MKQA_HF_NAME, split="train")
    rows = [dict(x) for x in ds]
    print(f"[mkqa] loaded from HF -> rows={len(rows)}")
    return rows

def _load_mkqa_from_raw() -> List[Dict[str, Any]]:
    import io
    import requests

    print("[mkqa] HF loading failed, fallback to raw GitHub gzip")
    response = requests.get(MKQA_RAW_URL, timeout=180)
    response.raise_for_status()
    rows = []
    with gzip.GzipFile(fileobj=io.BytesIO(response.content)) as gz:
        for line in gz:
            rows.append(json.loads(line.decode("utf-8")))
    print(f"[mkqa] loaded from raw -> rows={len(rows)}")
    return rows

def load_mkqa() -> List[Dict[str, Any]]:
    try:
        return _load_mkqa_from_hf()
    except Exception as e:
        print(f"[mkqa] HF load failed: {type(e).__name__}: {e}")
        return _load_mkqa_from_raw()

mkqa_raw = load_mkqa()

[mkqa] trying Hugging Face dataset: apple/mkqa
[mkqa] HF load failed: RuntimeError: Dataset scripts are no longer supported, but found mkqa.py
[mkqa] HF loading failed, fallback to raw GitHub gzip
[mkqa] loaded from raw -> rows=10000


In [8]:

# =========================
# НОРМАЛИЗАЦИЯ ANSWERS
# =========================

def normalize_rubq_answer(ans: Dict[str, Any]) -> Dict[str, Any]:
    answer_type = normalize_spaces(ans.get("type"))
    qid = extract_qid_from_uri(ans.get("value")) if answer_type == "uri" else None
    wd_names = ans.get("wd_names") or {}
    aliases = flatten_str_list([
        ans.get("label"),
        wd_names.get("ru", []),
        wd_names.get("en", []),
        ans.get("wp_names", []),
    ])
    return {
        "text": normalize_spaces(ans.get("label") if answer_type == "uri" else ans.get("value")),
        "normalized_text": normalize_for_dedup(ans.get("label") if answer_type == "uri" else ans.get("value")),
        "entity_id": qid,
        "answer_type": "entity" if answer_type == "uri" else "literal",
        "datatype": ans.get("datatype"),
        "xml_lang": ans.get("xml:lang"),
        "aliases": aliases,
        "source_names": {
            "wd_names": ans.get("wd_names"),
            "wp_names": ans.get("wp_names"),
        },
    }

def mkqa_answer_type_to_name(value: Any) -> str:
    if isinstance(value, int):
        if 0 <= value < len(MKQA_ANSWER_TYPE_NAMES):
            return MKQA_ANSWER_TYPE_NAMES[value]
        return f"unknown_{value}"
    return normalize_spaces(value)

def normalize_mkqa_answer(ans: Dict[str, Any]) -> Dict[str, Any]:
    answer_type = mkqa_answer_type_to_name(ans.get("type"))
    entity_id = normalize_spaces(ans.get("entity")) or None
    aliases = flatten_str_list([ans.get("text"), ans.get("aliases", [])])

    return {
        "text": normalize_spaces(ans.get("text")),
        "normalized_text": normalize_for_dedup(ans.get("text")),
        "entity_id": entity_id if entity_id else None,
        "answer_type": answer_type,
        "datatype": None,
        "xml_lang": "ru",
        "aliases": aliases,
        "source_names": {
            "wd_names": {"ru": aliases} if aliases else {},
            "wp_names": [],
        },
    }

In [9]:

# =========================
# НОРМАЛИЗАЦИЯ RuBQ -> unified schema
# =========================

def build_base_record() -> Dict[str, Any]:
    return {
        "benchmark_id": None,
        "source_dataset": None,
        "source_split": None,
        "source_example_id": None,
        "import_mode": None,
        "use_role": None,
        "question_ru": None,
        "question_en_original": None,
        "domain": None,
        "reasoning_type": [],
        "hop_count": None,
        "constraints_struct": {},
        "formal_query": None,
        "program": None,
        "gold_answers": [],
        "gold_qids": [],
        "supporting_facts": [],
        "supporting_entities": [],
        "evidence_sources": [],
        "is_answerable": True,
        "expected_cardinality": None,
        "paraphrase_cluster_id": None,
        "difficulty": "unknown",
        "manual_audit_status": "pending",
        "source_metadata": {},
    }

def pick_rubq_role(row: Dict[str, Any]) -> Tuple[str, str]:
    tags = row.get("tags", []) or []
    is_complex = has_rubq_complexity(tags)
    if KEEP_RUBQ_COMPLEX_ONLY_FOR_DIRECT and is_complex:
        return "direct_ru", "direct_ru_candidate"
    if is_complex:
        return "direct_ru", "direct_ru_candidate"
    return "direct_ru", "wording_donor"

def normalize_rubq_record(row: Dict[str, Any], split: str) -> Dict[str, Any]:
    rec = build_base_record()
    import_mode, use_role = pick_rubq_role(row)

    question_ru = normalize_spaces(row.get("question_text"))

    gold_answers = [normalize_rubq_answer(a) for a in row.get("answers", [])]
    gold_qids = unique_preserve_order([a["entity_id"] for a in gold_answers if a.get("entity_id")])

    support_entities = []
    for uri in row.get("question_uris", []) or []:
        qid = extract_qid_from_uri(uri)
        if qid:
            support_entities.append(qid)
    for prop in row.get("question_props", []) or []:
        pid = extract_pid_from_prop(prop)
        if pid:
            support_entities.append(pid)

    tags = row.get("tags", []) or []
    reasoning_type = rubq_reasoning_types(tags)

    rec.update({
        "benchmark_id": f"ext_rubq2_{split}_{int(row['uid']):06d}",
        "source_dataset": "RuBQ_2.0",
        "source_split": split,
        "source_example_id": str(row.get("uid")),
        "import_mode": import_mode,
        "use_role": use_role,
        "question_ru": question_ru,
        "question_en_original": normalize_spaces(row.get("question_eng")),
        "reasoning_type": reasoning_type,
        "hop_count": rubq_hop_count(tags),
        "constraints_struct": {
            "question_uris": row.get("question_uris", []),
            "question_props": row.get("question_props", []),
            "rubq_tags": tags,
        },
        "formal_query": normalize_spaces(row.get("query")),
        "program": None,
        "gold_answers": gold_answers,
        "gold_qids": gold_qids,
        "supporting_facts": [],
        "supporting_entities": unique_preserve_order(support_entities),
        "evidence_sources": [
            {
                "source_name": "Wikidata",
                "source_type": "knowledge_graph",
                "formal_query_available": True,
            }
        ],
        "is_answerable": "no_answer" not in set(tags),
        "expected_cardinality": len(gold_answers),
        "source_metadata": {
            "answer_text_raw": row.get("answer_text"),
            "paragraphs_uids": row.get("paragraphs_uids"),
            "rubq_version": row.get("RuBQ_version"),
        },
    })
    return rec

rubq_normalized = []
for split_name, rows in rubq_raw.items():
    for row in rows:
        rubq_normalized.append(normalize_rubq_record(row, split=split_name))

print(f"[rubq] normalized rows: {len(rubq_normalized):,}")

[rubq] normalized rows: 2,910


In [10]:

# =========================
# НОРМАЛИЗАЦИЯ MKQA -> unified schema
# =========================

def get_mkqa_ru_question(row: Dict[str, Any]) -> str:
    queries = row.get("queries", {}) or {}
    return normalize_spaces(queries.get("ru"))

def get_mkqa_ru_answers(row: Dict[str, Any]) -> List[Dict[str, Any]]:
    answers = row.get("answers", {}) or {}
    ru_answers = answers.get("ru", []) or []
    return [normalize_mkqa_answer(a) for a in ru_answers]

def pick_mkqa_role(question_ru: str, answers_ru: List[Dict[str, Any]]) -> Tuple[str, str, int, List[str]]:
    reasoning_types = detect_mkqa_reasoning_types(question_ru)
    score = mkqa_complexity_score(question_ru, answers_ru)

    # MKQA в целом не содержит SPARQL и не является “готовым” multi-hop truth layer.
    # Поэтому по умолчанию это либо reconstruction_candidate, либо wording_donor.
    if score >= MIN_MKQA_COMPLEXITY_SCORE_FOR_RECONSTRUCTION:
        return "template_transfer", "reconstruction_candidate", score, reasoning_types
    return "paraphrase_transfer", "wording_donor", score, reasoning_types

def normalize_mkqa_record(row: Dict[str, Any]) -> Dict[str, Any]:
    rec = build_base_record()

    question_ru = get_mkqa_ru_question(row)
    answers_ru = get_mkqa_ru_answers(row)
    import_mode, use_role, score, reasoning_types = pick_mkqa_role(question_ru, answers_ru)

    gold_qids = unique_preserve_order([a["entity_id"] for a in answers_ru if a.get("entity_id")])
    support_entities = list(gold_qids)

    rec.update({
        "benchmark_id": f"ext_mkqa_train_{str(row.get('example_id'))}",
        "source_dataset": "MKQA",
        "source_split": "train",
        "source_example_id": str(row.get("example_id")),
        "import_mode": import_mode,
        "use_role": use_role,
        "question_ru": question_ru,
        "question_en_original": normalize_spaces(row.get("query")),
        "reasoning_type": reasoning_types,
        "hop_count": None,
        "constraints_struct": {
            "mkqa_complexity_score": score,
            "mkqa_answer_types": unique_preserve_order([a.get("answer_type") for a in answers_ru]),
        },
        "formal_query": None,
        "program": None,
        "gold_answers": answers_ru,
        "gold_qids": gold_qids,
        "supporting_facts": [],
        "supporting_entities": support_entities,
        "evidence_sources": [
            {
                "source_name": "MKQA",
                "source_type": "multilingual_open_domain_qa",
                "formal_query_available": False,
            }
        ],
        "is_answerable": not any(a.get("answer_type") == "unanswerable" for a in answers_ru),
        "expected_cardinality": len(answers_ru),
        "source_metadata": {
            "all_languages_present": sorted(list((row.get("queries") or {}).keys())),
            "mkqa_complexity_score": score,
        },
    })
    return rec

mkqa_normalized = []
for row in mkqa_raw:
    q_ru = get_mkqa_ru_question(row)
    if not q_ru:
        continue
    mkqa_normalized.append(normalize_mkqa_record(row))

print(f"[mkqa] normalized Russian rows: {len(mkqa_normalized):,}")

[mkqa] normalized Russian rows: 10,000


In [11]:
# =========================
# PASS-THROUGH IMPORT LAYER
# =========================

# В этой версии ноутбука мы специально НЕ делаем:
# - quality filtering
# - dedup внутри источников
# - dedup против core-layer
#
# Ты потом сможешь отдельно применить свои собственные правила
# санитичека и дедупликации уже на экспортированных файлах.

rubq_imported = list(rubq_normalized)
mkqa_imported = list(mkqa_normalized)

print(f"[pass-through] RuBQ imported rows: {len(rubq_imported):,}")
print(f"[pass-through] MKQA imported rows: {len(mkqa_imported):,}")


[pass-through] RuBQ imported rows: 2,910
[pass-through] MKQA imported rows: 10,000


In [12]:

# =========================
# ГРУППИРОВКА ПО РОЛЯМ
# =========================

def split_by_role(records: List[Dict[str, Any]]) -> Dict[str, List[Dict[str, Any]]]:
    buckets = defaultdict(list)
    for rec in records:
        buckets[rec.get("use_role", "unknown")].append(rec)
    return dict(buckets)

rubq_by_role = split_by_role(rubq_imported)
mkqa_by_role = split_by_role(mkqa_imported)

for name, bucket in rubq_by_role.items():
    print(f"[rubq role] {name}: {len(bucket):,}")

for name, bucket in mkqa_by_role.items():
    print(f"[mkqa role] {name}: {len(bucket):,}")

[rubq role] wording_donor: 2,357
[rubq role] direct_ru_candidate: 553
[mkqa role] wording_donor: 7,851
[mkqa role] reconstruction_candidate: 2,149


In [13]:

# =========================
# ЭКСПОРТ
# =========================

def export_role_bucket(records: List[Dict[str, Any]], name: str) -> None:
    jsonl_path = EXPORT_JSONL_DIR / f"{name}.jsonl"
    csv_path = EXPORT_CSV_DIR / f"{name}.csv"

    write_jsonl(records, jsonl_path)

    csv_rows = []
    for rec in records:
        csv_rows.append({
            "benchmark_id": rec.get("benchmark_id"),
            "source_dataset": rec.get("source_dataset"),
            "source_split": rec.get("source_split"),
            "source_example_id": rec.get("source_example_id"),
            "import_mode": rec.get("import_mode"),
            "use_role": rec.get("use_role"),
            "question_ru": rec.get("question_ru"),
            "question_en_original": rec.get("question_en_original"),
            "reasoning_type": ", ".join(rec.get("reasoning_type", [])),
            "hop_count": rec.get("hop_count"),
            "expected_cardinality": rec.get("expected_cardinality"),
            "is_answerable": rec.get("is_answerable"),
            "gold_qids": ", ".join(rec.get("gold_qids", [])),
            "formal_query": rec.get("formal_query"),
        })
    save_csv(csv_rows, csv_path)
    print(f"[export] {name} -> {jsonl_path.name}, {csv_path.name}")

# RuBQ
for role_name, records in rubq_by_role.items():
    export_role_bucket(records, f"rubq2_{role_name}")

# MKQA
for role_name, records in mkqa_by_role.items():
    export_role_bucket(records, f"mkqa_{role_name}")

# merged stage-1
merged_stage1 = rubq_imported + mkqa_imported
export_role_bucket(merged_stage1, "external_stage1_merged")

[export] rubq2_wording_donor -> rubq2_wording_donor.jsonl, rubq2_wording_donor.csv
[export] rubq2_direct_ru_candidate -> rubq2_direct_ru_candidate.jsonl, rubq2_direct_ru_candidate.csv
[export] mkqa_wording_donor -> mkqa_wording_donor.jsonl, mkqa_wording_donor.csv
[export] mkqa_reconstruction_candidate -> mkqa_reconstruction_candidate.jsonl, mkqa_reconstruction_candidate.csv
[export] external_stage1_merged -> external_stage1_merged.jsonl, external_stage1_merged.csv


In [14]:

# =========================
# АУДИТ-ОЧЕРЕДИ
# =========================

def build_audit_queue(records: List[Dict[str, Any]], max_rows: int = 200) -> List[Dict[str, Any]]:
    # Приоритет:
    # 1) direct_ru_candidate
    # 2) reconstruction_candidate
    # 3) wording_donor
    role_priority = {
        "direct_ru_candidate": 0,
        "reconstruction_candidate": 1,
        "wording_donor": 2,
    }

    sorted_records = sorted(
        records,
        key=lambda r: (
            role_priority.get(r.get("use_role"), 99),
            -(r.get("constraints_struct", {}).get("mkqa_complexity_score", 0)),
            -(len(r.get("reasoning_type", []))),
            r.get("benchmark_id", ""),
        )
    )

    audit_rows = []
    for rec in sorted_records[:max_rows]:
        audit_rows.append({
            "benchmark_id": rec.get("benchmark_id"),
            "source_dataset": rec.get("source_dataset"),
            "source_split": rec.get("source_split"),
            "use_role": rec.get("use_role"),
            "import_mode": rec.get("import_mode"),
            "question_ru": rec.get("question_ru"),
            "question_en_original": rec.get("question_en_original"),
            "reasoning_type": ", ".join(rec.get("reasoning_type", [])),
            "expected_cardinality": rec.get("expected_cardinality"),
            "is_answerable": rec.get("is_answerable"),
            "gold_preview": " | ".join([a.get("text", "") for a in rec.get("gold_answers", [])[:5]]),
            "formal_query": rec.get("formal_query"),
        })
    return audit_rows

audit_queue = build_audit_queue(merged_stage1, max_rows=MAX_AUDIT_ROWS_PER_GROUP)
audit_queue_path = REPORTS_DIR / "audit_queue_stage1.csv"
save_csv(audit_queue, audit_queue_path)
print(f"[audit] saved: {audit_queue_path.name} ({len(audit_queue):,} rows)")

[audit] saved: audit_queue_stage1.csv (200 rows)


In [15]:

# =========================
# SUMMARY REPORTS
# =========================

def summarize(records: List[Dict[str, Any]], name: str) -> Dict[str, Any]:
    role_counter = Counter(r.get("use_role") for r in records)
    source_counter = Counter(r.get("source_dataset") for r in records)
    reasoning_counter = Counter()
    for r in records:
        for rt in r.get("reasoning_type", []):
            reasoning_counter[rt] += 1

    summary = {
        "name": name,
        "n_records": len(records),
        "by_role": dict(role_counter),
        "by_source": dict(source_counter),
        "by_reasoning_type": dict(reasoning_counter),
    }
    return summary

reports = {
    "rubq_summary": summarize(rubq_imported, "rubq_imported"),
    "mkqa_summary": summarize(mkqa_imported, "mkqa_imported"),
    "merged_summary": summarize(merged_stage1, "external_stage1_merged")
}

summary_path = REPORTS_DIR / "summary_stage1.json"
summary_path.write_text(json.dumps(reports, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"[report] saved summary: {summary_path.name}")
print(json.dumps(reports["merged_summary"], ensure_ascii=False, indent=2))

[report] saved summary: summary_stage1.json
{
  "name": "external_stage1_merged",
  "n_records": 12910,
  "by_role": {
    "wording_donor": 10208,
    "direct_ru_candidate": 553,
    "reconstruction_candidate": 2149
  },
  "by_source": {
    "RuBQ_2.0": 2910,
    "MKQA": 10000
  },
  "by_reasoning_type": {
    "single_hop": 1884,
    "reverse_relation": 35,
    "multi_constraint": 377,
    "multi_hop": 69,
    "ranking": 19,
    "qualifier_constraint": 26,
    "negation_or_exclusion": 150,
    "qualifier_answer": 6,
    "count": 574,
    "temporal_or_duration": 2074,
    "single_hop_or_simple_open_qa": 6272,
    "comparison_or_superlative": 965,
    "set_intersection_or_multi_constraint": 718
  }
}


In [16]:

# =========================
# БЫСТРЫЙ INSPECT
# =========================

def preview_records(records: List[Dict[str, Any]], n: int = 5, columns: Optional[List[str]] = None) -> pd.DataFrame:
    columns = columns or [
        "benchmark_id",
        "source_dataset",
        "use_role",
        "question_ru",
        "reasoning_type",
        "expected_cardinality",
        "is_answerable",
    ]
    rows = []
    for rec in records[:n]:
        row = {}
        for col in columns:
            val = rec.get(col)
            if isinstance(val, list):
                val = ", ".join(map(str, val))
            row[col] = val
        rows.append(row)
    return pd.DataFrame(rows)

print("=== RuBQ preview ===")
display(preview_records(rubq_imported, n=10))

print("=== MKQA preview ===")
display(preview_records(mkqa_imported, n=10))

=== RuBQ preview ===


,benchmark_id,source_dataset,use_role,question_ru,reasoning_type,expected_cardinality,is_answerable
0,ext_rubq2_dev_000004,RuBQ_2.0,wording_donor,Какой стране принадлежит знаменитый остров Пасхи?,single_hop,1,True
1,ext_rubq2_dev_000007,RuBQ_2.0,wording_donor,С какой музыкальной группой неразрывно связано...,single_hop,2,True
2,ext_rubq2_dev_000014,RuBQ_2.0,wording_donor,Где находится Летний сад?,single_hop,1,True
3,ext_rubq2_dev_000022,RuBQ_2.0,wording_donor,Какой город является столицей Туркмении?,single_hop,1,True
4,ext_rubq2_dev_000025,RuBQ_2.0,wording_donor,В каком городе издавалась с 1857 г. А. Герцено...,single_hop,2,True
5,ext_rubq2_dev_000031,RuBQ_2.0,wording_donor,В какой стране находится второй из самых высок...,single_hop,1,True
6,ext_rubq2_dev_000038,RuBQ_2.0,wording_donor,В каком виде спорта прославилась Курникова?,single_hop,1,True
7,ext_rubq2_dev_000040,RuBQ_2.0,wording_donor,В каком городе родился Вольфганг Амадей Моцарт?,single_hop,1,True
8,ext_rubq2_dev_000041,RuBQ_2.0,wording_donor,В каком городе убили Джона Леннона?,single_hop,1,True
9,ext_rubq2_dev_000047,RuBQ_2.0,wording_donor,Гражданином какой страны был изобретатель теле...,single_hop,1,True


=== MKQA preview ===


,benchmark_id,source_dataset,use_role,question_ru,reasoning_type,expected_cardinality,is_answerable
0,ext_mkqa_train_3051930912491995402,MKQA,wording_donor,как долго строились башни-близнецы,single_hop_or_simple_open_qa,1,True
1,ext_mkqa_train_6277735658261425592,MKQA,wording_donor,откуда взялась фраза great scott,single_hop_or_simple_open_qa,1,True
2,ext_mkqa_train_-4766029169354683727,MKQA,wording_donor,кто поёт love you like there's no tomorrow,single_hop_or_simple_open_qa,1,True
3,ext_mkqa_train_-5007314435785513023,MKQA,wording_donor,сколкьо всего сезонов в королевах крика,single_hop_or_simple_open_qa,1,True
4,ext_mkqa_train_7960403399608384663,MKQA,reconstruction_candidate,когда в последний раз the lakers вышли в плейофф,"comparison_or_superlative, temporal_or_duration",1,True
5,ext_mkqa_train_-6261874607090673789,MKQA,wording_donor,сколько серий в сезоне очень странных дел,count,1,True
6,ext_mkqa_train_563260143484355911,MKQA,wording_donor,кто поет i hear you knocking but you can't com...,single_hop_or_simple_open_qa,1,True
7,ext_mkqa_train_5144765284694516156,MKQA,wording_donor,кто придумал шоу голос,single_hop_or_simple_open_qa,1,True
8,ext_mkqa_train_-2985814541728632412,MKQA,reconstruction_candidate,когда начался и закончился период Дикого Запада,"temporal_or_duration, set_intersection_or_mult...",2,True
9,ext_mkqa_train_8201177426304478596,MKQA,wording_donor,откуда логотип Старбакса произошел,single_hop_or_simple_open_qa,1,True


In [17]:

# =========================
# ПОЛЕЗНЫЕ ВЫБОРКИ ДЛЯ СЛЕДУЮЩИХ НОУТБУКОВ
# =========================

rubq_direct_candidates = [r for r in rubq_imported if r["use_role"] == "direct_ru_candidate"]
mkqa_reconstruction_candidates = [r for r in mkqa_imported if r["use_role"] == "reconstruction_candidate"]
mkqa_wording_donors = [r for r in mkqa_imported if r["use_role"] == "wording_donor"]

print(f"RuBQ direct candidates: {len(rubq_direct_candidates):,}")
print(f"MKQA reconstruction candidates: {len(mkqa_reconstruction_candidates):,}")
print(f"MKQA wording donors: {len(mkqa_wording_donors):,}")

# Отдельные удобные экспорты
write_jsonl(rubq_direct_candidates, EXPORT_JSONL_DIR / "rubq2_direct_candidates_only.jsonl")
write_jsonl(mkqa_reconstruction_candidates, EXPORT_JSONL_DIR / "mkqa_reconstruction_candidates_only.jsonl")
write_jsonl(mkqa_wording_donors, EXPORT_JSONL_DIR / "mkqa_wording_donors_only.jsonl")
print("[export] wrote focused candidate files.")

RuBQ direct candidates: 553
MKQA reconstruction candidates: 2,149
MKQA wording donors: 7,851
[export] wrote focused candidate files.


## Что делать после этого ноутбука

Следующий хороший шаг:

### Вариант A — сразу продолжить по плану
Сделать `03_kqapro_mintaka_qawiki_template_mining.ipynb`, где:
- выделяются **question families**,
- сохраняются **logical forms / programs / SPARQL patterns**,
- строится каталог новых типов русских multi-hop запросов.

### Вариант B — сначала отдельно прогнать твой собственный постпроцессинг
1. Сделать dedup так, как удобно именно тебе.
2. Добавить свои правила санитичека и аудита.
3. Разметить `domain` и `difficulty`.
4. Отфильтровать только те записи, которые реально полезны для расширения мастер-бенчмарка.

### Практическое правило
- **RuBQ 2.0** здесь — почти готовый внешний русскоязычный слой.
- **MKQA** здесь — в основном сырьё для:
  - живых русских формулировок,
  - нормализации answer aliases,
  - реконструкции под твой truth layer.


## Короткий чек-лист качества перед merge в мастер-бенч

- [ ] Вопрос звучит естественно по-русски.
- [ ] Пример действительно полезен для твоего benchmark-фокуса.
- [ ] Если это direct import, gold достаточно надёжен и интерпретируем.
- [ ] Если это reconstruction candidate, для него понятно, как построить formal query / constraints.
- [ ] Reasoning type размечен корректно.
- [ ] Нет явных shortcut-артефактов.
- [ ] Дедуп и санитичек уже проведены в отдельном шаге.
